In [0]:
%sql
CREATE VOLUME IF NOT EXISTS delivery.bronze.checkpoints;

In [0]:
from pyspark.sql import functions as F

conn = dbutils.secrets.get("kv-scope", "eh-listen-conn")
BOOTSTRAP = "evhns-delivery-1.servicebus.windows.net:9093"
jaas = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
    f'username="$ConnectionString" password="{conn}";'
)

raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas)
    .load()
)

bronze = raw.select(
    F.col("value").cast("string").alias("raw_payload"),
    "topic", "partition", "offset",
    F.col("timestamp").alias("kafka_timestamp"),
    F.current_timestamp().alias("ingested_at"),
)

(
    bronze.writeStream.format("delta")
    .option("checkpointLocation", "/Volumes/delivery/bronze/checkpoints/orders_raw")
    .trigger(availableNow=True)
    .toTable("delivery.bronze.orders_raw")
    .awaitTermination()
)

In [0]:
%sql
SELECT count(*) AS rows, count(DISTINCT offset) AS distinct_offsets FROM delivery.bronze.orders_raw;

SELECT raw_payload, partition, offset, ingested_at FROM delivery.bronze.orders_raw LIMIT 10;

SELECT count(*) AS row_count FROM delivery.bronze.orders_raw;